# Holzinger plasmode on Hoffman2

Use this notebook **inside a Hoffman2 Jupyter session** started with `h2jupynb` from your laptop
(Slurm host: `hoffman2.oarc.ucla.edu`). Setup notes: `cluster/hoffman2/README.md`.

**DGP knobs (fixed here):**
- Holzinger FA params from `params/plasmode_holzinger/`
- `cov_XZ = 0.9 * max_equal_cov_xz` (PD joint Σ)
- mean = 2, propensity `logit(π)=-0.6+Z`, 3-factor Semopy CFA

For long `R=1000` grids, prefer `sbatch` (`cluster/hoffman2/run_holzinger_array.sbatch`) instead of keeping this notebook open.

## 0. Paths + imports

Edit `REPO` if the notebook cwd is not the repo (or a subfolder).

In [ ]:
import json
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Default Hoffman layout for jspurgia; override with MISSING_DATA_2_ROOT if needed.
_default_repo = Path.home() / "research" / "missing_data_2"
REPO = Path(os.environ.get("MISSING_DATA_2_ROOT", _default_repo if _default_repo.exists() else Path.cwd())).resolve()
if not (REPO / "scripts" / "simulation.py").exists():
    if (REPO.parent / "scripts" / "simulation.py").exists():
        REPO = REPO.parent
    else:
        raise FileNotFoundError(
            f"Cannot find repo scripts under {REPO}. Set MISSING_DATA_2_ROOT or cd to the repo."
        )

sys.path.insert(0, str(REPO / "scripts"))
sys.path.insert(0, str(REPO))

from metrics import summarize_run
from misspec import available_scenarios
from plasmode import DEFAULT_PARAMS_DIR, load_plasmode_joint
from simulation import RunConfig, run_simulation

print("host:", os.uname().nodename if hasattr(os, "uname") else os.environ.get("HOSTNAME"))
print("repo:", REPO)
print("params:", DEFAULT_PARAMS_DIR)
print("params exist:", (DEFAULT_PARAMS_DIR / "lambda.csv").exists())

## 1. Load FA params → joint Σ (`cov_XZ = 0.9 × max`)

In [ ]:
plas = load_plasmode_joint(
    DEFAULT_PARAMS_DIR,
    cov_XZ=None,              # => 0.9 * max_equal_cov_xz
    cov_xz_frac_of_max=0.9,
    var_Z=1.0,
    mean_value=2.0,
)

print(f"p={plas['p']}  factors={plas['n_factors']}  cov_XZ={plas['cov_XZ']:.6f}  max={plas['cov_XZ_max']:.6f}")
print("min eig joint Σ:", float(np.linalg.eigvalsh(plas["Sigma"]).min()))
assert np.linalg.eigvalsh(plas["Sigma"]).min() > 0, "joint Sigma must be PD"

display(pd.DataFrame(plas["Lambda"], index=plas["variables"], columns=plas["factors"]).round(4))

## 2. Simulation config

Start with a smoke (`reps=2`). For the board study set `reps=1000` and expect many hours — use Slurm batch for that.

In [ ]:
n = [250, 500, 1000]   # board grid; use [250] to smoke one size
scenarios = list(available_scenarios())

reps = 2               # smoke; set 1000 for full
seed = 20261123
save_data = False
run_id_prefix = "holzinger"

scenarios, n, reps

## 3. Run

Each `(scenario, n)` writes under `runs/<run_id_prefix>_<scenario>_n<N>/` without overwriting.

In [ ]:
run_dirs = []

for scenario in scenarios:
    cfg = RunConfig(
        n=n,
        reps=reps,
        Sigma=plas["Sigma"],
        mean=plas["mean"],
        n_z=1,
        seed=seed,
        scenario=scenario,
        methods=("aipw",),
        propensity_intercept=-0.6,
        propensity_coef=1.0,
        use_true_pi=False,
        save_data=save_data,
        cfa_n_factors=plas["cfa_n_factors"],
        run_id=f"{run_id_prefix}_{scenario}",
    )
    out = run_simulation(cfg)
    if not isinstance(out, list):
        out = [out]
    run_dirs.extend(out)
    for d in out:
        print(scenario, "->", d)

len(run_dirs), run_dirs[:3]

## 4. Step-5 board metrics

In [ ]:
rows = []
for run_dir in run_dirs:
    summary = summarize_run(run_dir, write=True)
    overall = summary["overall_table"].copy()
    params = json.loads((run_dir / "params.json").read_text())
    overall.insert(0, "run_id", run_dir.name)
    overall.insert(1, "n", int(params["n"]))
    overall.insert(2, "scenario", params.get("scenario"))
    rows.append(overall)

results = pd.concat(rows, ignore_index=True)
board = results[[
    c for c in [
        "run_id", "n", "scenario", "method",
        "cov_rel_bias", "cov_mean_coverage95", "cov_mean_var_ratio",
        "psd_rate", "heywood_rate",
    ] if c in results.columns
]].copy()
if "psd_rate" in board.columns:
    board["prop_non_pd"] = 1.0 - board["psd_rate"]

out_csv = REPO / "runs" / f"{run_id_prefix}_board_metrics.csv"
board.to_csv(out_csv, index=False)
print("wrote", out_csv)
board

## 5. Optional: submit the same study as a Slurm batch job

Only works if this kernel can call `sbatch` (compute/login with Slurm). Usually you submit from an SSH shell instead.

In [ ]:
# Uncomment to submit the array job from the notebook environment.
# import subprocess
# cmd = [
#     "sbatch",
#     f"--export=ALL,N_LIST={','.join(map(str, n))},REPS=1000,COV_XZ_FRAC=0.9",
#     str(REPO / "cluster" / "hoffman2" / "run_holzinger_array.sbatch"),
# ]
# print(" ".join(cmd))
# subprocess.run(cmd, cwd=str(REPO), check=True)